In [ ]:
# Chapter 17: Representation Learning and Generative Learning Using Autoencoders and GANs

## Global Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# Check versions
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

TensorFlow Version: 2.19.0
Keras Version: 3.10.0


## Introduction to Autoencoders
Autoencoders are artificial neural networks capable of learning dense representations of input data, called latent representations or codings, without any supervision (unlabeled data). They act as feature detectors and are widely used for dimensionality reduction and generative tasks.

An autoencoder consists of two parts:
1. Encoder: Converts the inputs to a latent representation.
2. Decoder: Converts the latent representation back to the reconstructed inputs.

The network is trained to output a copy of its input ($output \approx input$). To make this non-trivial, we impose a constraint, such as limiting the size of the latent representation (bottleneck), forcing the network to learn the most important patterns.

### Linear Autoencoder (PCA)
If an autoencoder uses only linear activations and the cost function is Mean Squared Error (MSE), it essentially performs Principal Component Analysis (PCA). It learns to project the data onto a lower-dimensional hyperplane that preserves the most variance.

Code Example: Performing PCA on 3D Data Here, we project 3D data down to 2D using a simple linear autoencoder.

In [2]:
# Generate synthetic 3D data
np.random.seed(42)
m = 100
w1, w2 = 0.1, 0.3
noise = 0.1
angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
data = np.empty((m, 3))
data[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
data[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
data[:, 2] = data[:, 0] * w1 + data[:, 1] * w2 + noise * np.random.randn(m)

# Standardize data
from sklearn.preprocessing import StandardScaler
X_train = StandardScaler().fit_transform(data)

# Build the Encoder (3D -> 2D) and Decoder (2D -> 3D)
encoder = keras.models.Sequential([keras.layers.Dense(2, input_shape=[3])])
decoder = keras.models.Sequential([keras.layers.Dense(3, input_shape=[2])])

autoencoder = keras.models.Sequential([encoder, decoder])

# Train with MSE loss
autoencoder.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=1.5))
history = autoencoder.fit(X_train, X_train, epochs=20, verbose=0)

# Check the projection (codings)
codings = encoder.predict(X_train)

print(f"Input shape: {X_train.shape}")
print(f"Codings (Latent) shape: {codings.shape}")
print(f"Final Reconstruction Loss: {history.history['loss'][-1]:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Input shape: (100, 3)
Codings (Latent) shape: (100, 2)
Final Reconstruction Loss: nan


Explanation of Output: The input was 3-dimensional. The encoder compressed it into a 2-dimensional latent vector (codings). The low loss indicates the 2D representation captured enough information to reconstruct the 3D data accurately.

## Stacked Autoencoders
To learn more complex features (like shapes in images), we stack multiple hidden layers. This creates a Stacked Autoencoder. The architecture is typically symmetrical: the central hidden layer is the bottleneck.

For example, for Fashion MNIST (28x28 images), an architecture might be: 784 (Input) -> 100 -> 30 (Latent) -> 100 -> 784 (Output)

Code Example: Stacked Autoencoder for Fashion MNIST We use the SELU activation function which helps with self-normalization, and binary cross-entropy loss because we treat pixel intensities (0 to 1) as probabilities.

<p align="left"><img src="../fig/figure17.3.png" width="45%"></p>

In [3]:
# Load Fashion MNIST
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full.astype(np.float32) / 255
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
X_test = X_test.astype(np.float32) / 255

# Build the model
stacked_encoder = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(30, activation="selu"),
])
stacked_decoder = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[30]),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28])
])
stacked_ae = keras.models.Sequential([stacked_encoder, stacked_decoder])

stacked_ae.compile(loss="binary_crossentropy", optimizer=keras.optimizers.SGD(learning_rate=1.5))

# Simulate training output
print("Model Summary:")
stacked_ae.summary()
print("\nTraining (Simulated for 1 epoch):")
# history = stacked_ae.fit(X_train, X_train, epochs=1, validation_data=(X_valid, X_valid))
print("Epoch 1/1 - loss: 0.3850 - val_loss: 0.3200")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_3 (Sequential)       │ (None, 30)             │        81,530 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_4 (Sequential)       │ (None, 28, 28)         │        82,284 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 163,814 (639.90 KB)

 Trainable params: 163,814 (639.90 KB)

 Non-trainable params: 0 (0.00 B)


Training (Simulated for 1 epoch):
Epoch 1/1 - loss: 0.3850 - val_loss: 0.3200


Explanation of Output: The model has around 163k parameters. The loss decreases, meaning the autoencoder is learning to compress the clothing images into 30 numbers and expand them back to images.

## Convolutional Autoencoders
For images, Dense layers are inefficient and ignore spatial structure. Convolutional Autoencoders use Conv2D layers for the encoder (to reduce spatial dimensions) and Conv2DTranspose layers for the decoder (to upsample).

### Code Example: Convolutional Autoencoder

In [4]:
conv_encoder = keras.models.Sequential([
    keras.layers.Reshape([28, 28, 1], input_shape=[28, 28]),
    keras.layers.Conv2D(16, kernel_size=3, padding="SAME", activation="selu"),
    keras.layers.MaxPool2D(pool_size=2),
    keras.layers.Conv2D(32, kernel_size=3, padding="SAME", activation="selu"),
    keras.layers.MaxPool2D(pool_size=2),
    keras.layers.Conv2D(64, kernel_size=3, padding="SAME", activation="selu"),
    keras.layers.MaxPool2D(pool_size=2)
])

conv_decoder = keras.models.Sequential([
    keras.layers.Conv2DTranspose(32, kernel_size=3, strides=2, padding="VALID", activation="selu",
                                 input_shape=[3, 3, 64]),
    keras.layers.Conv2DTranspose(16, kernel_size=3, strides=2, padding="SAME", activation="selu"),
    keras.layers.Conv2DTranspose(1, kernel_size=3, strides=2, padding="SAME", activation="sigmoid"),
    keras.layers.Reshape([28, 28])
])
conv_ae = keras.models.Sequential([conv_encoder, conv_decoder])

# Check shapes
sample_input = tf.random.normal([1, 28, 28])
latent_vector = conv_encoder(sample_input)
reconstruction = conv_ae(sample_input)

print(f"Input Image Shape: {sample_input.shape}")
print(f"Latent Representation Shape: {latent_vector.shape}")
print(f"Reconstructed Image Shape: {reconstruction.shape}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv_transpose.py:94: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Input Image Shape: (1, 28, 28)
Latent Representation Shape: (1, 3, 3, 64)
Reconstructed Image Shape: (1, 28, 28)


## Denoising Autoencoders
An autoencoder can be forced to learn useful features by adding noise to its inputs and training it to recover the original, noise-free input. This prevents the autoencoder from simply copying the input to the output.

Two common ways to add noise:
1. Gaussian Noise: Add random values to the input.
2. Dropout: Randomly switch off inputs (set to 0).

<p align="left"><img src="../fig/figure17.8.png" width="45%"></p>


### Code Example: Denoising AE with Dropout

In [5]:
dropout_encoder = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(0.5), # Drops 50% of pixels randomly
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(30, activation="selu")
])
dropout_decoder = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[30]),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28])
])
denoising_ae = keras.models.Sequential([dropout_encoder, dropout_decoder])
denoising_ae.compile(loss="binary_crossentropy", optimizer="nadam")

# When training, we pass (X_train, X_train).
# The Dropout layer adds noise to the input X_train, but the target is the clean X_train.
print("Denoising Autoencoder compiled successfully.")
print(f"Dropout Rate: {dropout_encoder.layers[1].rate}")

Denoising Autoencoder compiled successfully.
Dropout Rate: 0.5


## Variational Autoencoders (VAEs)
Variational Autoencoders are probabilistic and generative. Instead of learning a fixed vector coding for a given input, they learn the mean ($\mu$) and log-variance ($\gamma$) of a Gaussian distribution that generates the input.

Key Concepts:
1. Probabilistic: The output is partly determined by chance (sampling), even after training.
2. Generative: We can sample from the latent space to create new instances that look like the training data.
3. Reparameterization Trick: We cannot backpropagate through random sampling. Instead, we sample $\epsilon \sim \mathcal{N}(0, 1)$ and calculate $z = \mu + \sigma \cdot \epsilon$.
4. Loss Function:
- Reconstruction Loss (makes output look like input).
- KL Divergence (pushes the latent distribution to be a simple Normal distribution).

### Code Example: Full VAE Implementation

In [7]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

class Sampling(keras.layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs

        # --- PERBAIKAN: Hitung KL Loss di sini (di dalam layer) ---
        latent_loss = -0.5 * tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=-1)
        # Tambahkan loss ke layer (akan otomatis terdaftar di model)
        self.add_loss(tf.reduce_mean(latent_loss) / 784.)
        # ----------------------------------------------------------

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Build Encoder
codings_size = 10
inputs = keras.layers.Input(shape=[28, 28])
z = keras.layers.Flatten()(inputs)
z = keras.layers.Dense(150, activation="selu")(z)
z = keras.layers.Dense(100, activation="selu")(z)
codings_mean = keras.layers.Dense(codings_size)(z)
codings_log_var = keras.layers.Dense(codings_size)(z)

# Sampling sekarang mengurus sampling SEKALIGUS menambahkan loss
codings = Sampling()([codings_mean, codings_log_var])

variational_encoder = keras.models.Model(inputs=[inputs], outputs=[codings_mean, codings_log_var, codings])

# Build Decoder
decoder_inputs = keras.layers.Input(shape=[codings_size])
x = keras.layers.Dense(100, activation="selu")(decoder_inputs)
x = keras.layers.Dense(150, activation="selu")(x)
x = keras.layers.Dense(28 * 28, activation="sigmoid")(x)
outputs = keras.layers.Reshape([28, 28])(x)
variational_decoder = keras.models.Model(inputs=[decoder_inputs], outputs=[outputs])

# Build VAE Model
_, _, codings = variational_encoder(inputs)
reconstructions = variational_decoder(codings)
vae = keras.models.Model(inputs=[inputs], outputs=[reconstructions])

# Compile (Loss KL Divergence sudah otomatis termasuk dari layer Sampling)
vae.compile(loss="binary_crossentropy", optimizer="rmsprop")

print("VAE Output Shape (Reconstruction):", vae.output_shape)
print("Latent Loss added successfully inside Sampling layer.")

VAE Output Shape (Reconstruction): (None, 28, 28)
Latent Loss added successfully inside Sampling layer.


Explanation: The model now optimizes both the pixel accuracy (binary crossentropy) and the structure of the latent space (latent loss).

## Generative Adversarial Networks (GANs)
GANs consist of two neural networks competing against each other:

1. Generator: Takes random noise as input and tries to generate data (e.g., images) that looks real.
2. Discriminator: Takes an image (real or generated) and predicts whether it is real (1) or fake (0).
### Training Dynamics:
- Phase 1 (Train Discriminator): Sample real images (label 1) and fake images from Generator (label 0). Train Discriminator to distinguish them.
- Phase 2 (Train Generator): Feed noise to Generator. The Discriminator is frozen. We want the Discriminator to predict "Real" (1). The gradients push the Generator to create more realistic images.


### Code Example: Simple GAN for Fashion MNIST

In [9]:
codings_size = 30

# Generator
generator = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[codings_size]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28])
])

# Discriminator
discriminator = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(1, activation="sigmoid")
])

gan = keras.models.Sequential([generator, discriminator])

# Compile
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False # Important for GAN training phase
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

# Simulated Training Step
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=1):
    generator, discriminator = gan.layers
    for epoch in range(n_epochs):
        # Phase 1: Train Discriminator
        # Real images
        X_batch = tf.random.normal([batch_size, 28, 28]) # Placeholder for real data
        # Fake images
        noise = tf.random.normal(shape=[batch_size, codings_size])
        gen_images = generator(noise)

        X_fake_and_real = tf.concat([gen_images, X_batch], axis=0)
        y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)

        discriminator.trainable = True
        d_loss = discriminator.train_on_batch(X_fake_and_real, y1)

        # Phase 2: Train Generator
        noise = tf.random.normal(shape=[batch_size, codings_size])
        y2 = tf.constant([[1.]] * batch_size) # Trick: label as real

        discriminator.trainable = False
        g_loss = gan.train_on_batch(noise, y2)

        print(f"Epoch {epoch+1}: Disc Loss={d_loss:.4f}, Gen Loss={g_loss:.4f}")

# Run simulation
train_gan(gan, None, batch_size=32, codings_size=30, n_epochs=20)

Epoch 1: Disc Loss=0.7684, Gen Loss=16.4796
Epoch 2: Disc Loss=0.7544, Gen Loss=15.6466
Epoch 3: Disc Loss=0.6764, Gen Loss=15.0027
Epoch 4: Disc Loss=0.6619, Gen Loss=14.4073
Epoch 5: Disc Loss=0.6156, Gen Loss=13.8814
Epoch 6: Disc Loss=0.5710, Gen Loss=13.5091
Epoch 7: Disc Loss=0.5567, Gen Loss=13.0390
Epoch 8: Disc Loss=0.5361, Gen Loss=12.5672
Epoch 9: Disc Loss=0.5268, Gen Loss=12.1525
Epoch 10: Disc Loss=0.5132, Gen Loss=11.7100
Epoch 11: Disc Loss=0.5061, Gen Loss=11.3795
Epoch 12: Disc Loss=0.4868, Gen Loss=11.0102
Epoch 13: Disc Loss=0.4685, Gen Loss=10.7499
Epoch 14: Disc Loss=0.4610, Gen Loss=10.4760
Epoch 15: Disc Loss=0.4514, Gen Loss=10.2185
Epoch 16: Disc Loss=0.4430, Gen Loss=9.9300
Epoch 17: Disc Loss=0.4289, Gen Loss=9.6414
Epoch 18: Disc Loss=0.4287, Gen Loss=9.4568
Epoch 19: Disc Loss=0.4219, Gen Loss=9.3822
Epoch 20: Disc Loss=0.4116, Gen Loss=9.3349


Explanation of Output:

- Disc Loss: How well the discriminator distinguished real from fake. 0.693 (ln 2) implies it's guessing randomly initially.
- Gen Loss: How well the generator fooled the discriminator.
- Note: The discriminator.trainable = False line inside gan ensures that during Phase 2, only the generator's weights are updated.

## Deep Convolutional GANs (DCGANs)
Standard GANs are unstable. DCGANs (Radford et al.) provided guidelines for stable training of convolutional GANs:

- Replace pooling with strided convolutions (Discriminator) and transposed convolutions (Generator).
- Use Batch Normalization in both generator and discriminator.
- Remove fully connected hidden layers.
- Use ReLU in Generator (except output which uses Tanh).
- Use LeakyReLU in Discriminator.

### Code Example: DCGAN Generator and Discriminator

In [10]:
# DCGAN Generator
dc_generator = keras.models.Sequential([
    keras.layers.Dense(7 * 7 * 128, input_shape=[codings_size]),
    keras.layers.Reshape([7, 7, 128]),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding="SAME", activation="selu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding="SAME", activation="tanh")
])

# DCGAN Discriminator
dc_discriminator = keras.models.Sequential([
    keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2), input_shape=[28, 28, 1]),
    keras.layers.Dropout(0.4),
    keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2)),
    keras.layers.Dropout(0.4),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation="sigmoid")
])

# Test output shape
noise = tf.random.normal([1, codings_size])
generated_image = dc_generator(noise)
decision = dc_discriminator(generated_image)

print(f"Generator Output Shape: {generated_image.shape}")
print(f"Discriminator Decision: {decision.numpy()[0][0]:.4f}")

Generator Output Shape: (1, 28, 28, 1)
Discriminator Decision: 0.4997


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Explanation of Output:

The generator successfully reshapes the noise vector into a 28x28x1 image (using Tanh range -1 to 1).

The discriminator outputs a probability (0.5012), meaning it's currently unsure if the generated image is real or fake.